# DAEN 328 Assignment 5: Load and Serve

**Name:** _your name here_

Save this notebook as `A5_LastName_FirstName.ipynb`. It must run **top to bottom** (Kernel > Restart & Run All) with all outputs showing when you submit.

Work through `Assignment05_Guide.ipynb` first. Every technique you need is taught there on practice data. This notebook is where you apply it to `EmployeeDB`.

Your dashboard (Part C) is a separate file, `app.py`. See the instructions PDF.

## CONFIG: edit this cell once

In [26]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "ADO7104v"      # change to YOUR PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"

CSV_FILE = "Cleaned_data.csv"   # your Assignment 3 output, in the same folder as this notebook

DB_PARAMS = {"dbname": DB_NAME, "user": DB_USER, "password": DB_PASSWORD,
             "host": DB_HOST, "port": DB_PORT}
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Part A: Load your cleaned CSV into the 7 EmployeeDB tables

### A1. Read and inspect

Read the CSV with every column as text. Show its shape and the number of blanks per column.

In [27]:
raw = pd.read_csv(CSV_FILE, dtype=str)

print("Dataset dimensions:", raw.shape)

print("\nMissing values per column:")
print(raw.isna().sum())

raw.head()

Dataset dimensions: (14321, 24)

Missing values per column:
EmployeeID               0
Name                     0
HighestDegree            0
Division                 0
DivisionAddress          0
DivisionPhone            0
DivStreetAddress         0
DivCity                  0
DivState                 0
DivZip                   0
PersonalAddress          0
PerStreetAddress         0
PerCity                  0
PerState                 0
PerZip                   0
Email                    0
PhoneNumber              0
Department               0
DepartmentDescription    0
JobTitle                 0
JobDescription           0
Salary                   0
Birthdate                0
SocialSecurity           0
dtype: int64


,EmployeeID,Name,HighestDegree,Division,DivisionAddress,DivisionPhone,DivStreetAddress,DivCity,DivState,DivZip,...,PerZip,Email,PhoneNumber,Department,DepartmentDescription,JobTitle,JobDescription,Salary,Birthdate,SocialSecurity
0,436822,Steven Sean Johnson,MA,South Division,"456 South Ave, Austin, TX 73301",(512)555-6789,456 South Ave,Austin,TX,73301,...,86189,steven_johnson_145@KLB.com,(983)309-3096,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 2,Provides administrative support within the Sou...,43996.06,12/10/1971,149-51-0794
1,224873,Robert Matthew Schultz,CTE,North Division,"123 North St, Springfield, IL 62701",(217)555-1234,123 North St,Springfield,IL,62701,...,81132,robert_schultz_117@KLB.com,(696)415-7443,Operations,"Manage all purchasing, manufacturing, and logi...",Operations Associate Level 4,Performs production and operational tasks with...,37820.64,01/18/2005,097-83-2017
2,494157,Donna Brittany Ford,AB,North Division,"123 North St, Springfield, IL 62701",(217)555-1234,123 North St,Springfield,IL,62701,...,44937,donna_ford_151@KLB.com,(217)555-1234,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 5,Provides administrative support within the Nor...,47606.65,02/13/1965,900-00-0000
3,460666,Teresa Jennifer Hernandez,THD,South Division,"456 South Ave, Austin, TX 73301",(512)555-6789,456 South Ave,Austin,TX,73301,...,17454,teresa_hernandez_162@KLB.com,(894)497-2204,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 5,Provides administrative support within the Sou...,41088.33,05/21/1975,421-73-2111
4,471359,Christopher Perry Lopez,AB,West Division,"101 West Dr, Sacramento, CA 94203",(916)555-5678,101 West Dr,Sacramento,CA,94203,...,70726,christopher_lopez_100@KLB.com,(859)491-9776,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 6,Provides administrative support within the Wes...,45006.0,10/01/1973,343-92-4420


### A2. Clean and validate in pandas

Required:
- Strip spaces from text columns.
- Convert `Birthdate` to a real date (`pd.to_datetime(..., errors="coerce").dt.date`) and **print how many could not be read**.
- Convert `Salary` and `EmployeeID` to numbers.
- Put every row that cannot satisfy your schema's NOT NULL or CHECK constraints into a `rejects` DataFrame, save it as `rejects_employees.csv`, and print rows read / good / rejected. Never invent a value (such as a salary of 0) to get a row past a constraint.

In [28]:
# A2: Clean and validate employee data

# 1. Copy the original DataFrame
df = raw.copy()

# 2. Strip whitespace from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# 3. Convert Birthdate
df["Birthdate"] = pd.to_datetime(df["Birthdate"], format="%m/%d/%Y", errors="coerce").dt.date

bad_dates = df["Birthdate"].isna().sum()
print("Dates that couldn't be read:", bad_dates)

# 4. Convert Salary and EmployeeID
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["EmployeeID"] = pd.to_numeric(df["EmployeeID"], errors="coerce")

# 5. Check numeric and date constraints
invalid_salary = df["Salary"].isna() | (df["Salary"] <= 0)

invalid_id = (df["EmployeeID"].isna() | (df["EmployeeID"] % 1 != 0))

invalid_birthdate = df["Birthdate"].isna()

# 6. Check required text columns
# Verify these against your Assignment 4 NOT NULL constraints.
required_columns = ["Name", "SocialSecurity", "Division", "Department", "JobTitle"]

invalid_required = (df[required_columns].isna() | df[required_columns].eq("")).any(axis=1)

# 7. Check for duplicate EmployeeIDs
duplicate_ids = df["EmployeeID"].duplicated(keep="first")

# 8. Combine all invalid conditions
invalid_rows = (invalid_salary | invalid_id | invalid_birthdate | invalid_required | duplicate_ids)

# 9. Separate valid and rejected records
rejects = df[invalid_rows].copy()
good = df[~invalid_rows].copy()

# 10. Save rejected records
rejects.to_csv("rejects_employees.csv", index=False)

# 11. Display validation results
print("Invalid salaries:", invalid_salary.sum())
print("Invalid IDs:", invalid_id.sum())
print("Invalid birthdates:", invalid_birthdate.sum())
print("Missing required values:", invalid_required.sum())
print("Duplicate IDs:", duplicate_ids.sum())

print("\nRows read:", len(df))
print("Good records:", len(good))
print("Rejected records:", len(rejects))

print("All records accounted for:",
      len(good) + len(rejects) == len(df))


Dates that couldn't be read: 0
Invalid salaries: 0
Invalid IDs: 0
Invalid birthdates: 0
Missing required values: 0
Duplicate IDs: 0

Rows read: 14321
Good records: 14321
Rejected records: 0
All records accounted for: True


C:\Users\danie\AppData\Local\Temp\ipykernel_5584\403331246.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### Given: `clean()` (run it, do not change it)

Pass **every** value through `clean()` when you insert it. pandas marks a blank as `NaN`, and if a `NaN` reaches PostgreSQL it is stored as the value `NaN`, not as NULL. A `NaN` salary passes `NOT NULL` and `CHECK (salary > 0)` with no error, and then every average that includes it comes out as `NaN`. `clean()` turns blanks into `None` (a real NULL), so the database rejects what it should. It also turns numpy numbers into plain Python numbers, which avoids the error `can't adapt type numpy.int64`.

In [29]:
def clean(value):
    # Turn pandas blanks into None (SQL NULL) and numpy numbers into plain Python numbers.
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

### Given: start over with empty tables

While you build the loader, a bad attempt can leave wrong rows behind. To empty all 7 tables and restart the id numbers from 1, set `RESET = True`, run the cell, then set it back to `False`. This keeps your tables and constraints; it only removes the rows. Do not use `DROP TABLE`, which deletes your Assignment 4 schema.

Leave `RESET = False` when you submit.

In [30]:
RESET = False

if RESET:
    with engine.begin() as conn:
        conn.execute(text("""
            TRUNCATE employeedetails, personaldetails, jobdescriptions, divisionlocations,
                     departments, jobtitles, divisions
            RESTART IDENTITY CASCADE;
        """))
    print("All 7 tables emptied.")

### A3. The loader

Write `load_employees(good)`. Requirements:
- Loads all 7 tables, lookup tables first, `employeedetails` last, using the CSV-to-column map in the instructions PDF.
- Runs in **one transaction** (all or nothing).
- **Safe to re-run**: a second run adds zero rows to every table.
- Uses `%s` placeholders for every value, and passes every value through `clean()`.

In [37]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE jobtitles
        ALTER COLUMN jobtitle TYPE VARCHAR(255);
    """))

print("Job title column successfully updated!")

Job title column successfully updated!


In [31]:
with engine.connect() as conn:
    columns = pd.read_sql(text("""
        SELECT table_name, column_name
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name IN (
              'divisions', 'departments', 'jobtitles',
              'personaldetails', 'divisionlocations',
              'jobdescriptions', 'employeedetails'
          )
        ORDER BY table_name, ordinal_position;
    """), conn)

display(columns)

,table_name,column_name
0,departments,departmentid
1,departments,departmentname
2,departments,departmentdescription
3,divisionlocations,locationid
4,divisionlocations,divisionid
5,divisionlocations,divstreet
6,divisionlocations,divcity
7,divisionlocations,divstate
8,divisionlocations,divzip
9,divisionlocations,divisionphone


In [40]:
# A3: Load employee data into seven PostgreSQL tables

def load_employees(good):
    conn = psycopg2.connect(**DB_PARAMS)

    try:
        with conn:
            with conn.cursor() as cur:

                # 1. DIVISIONS
                for name in good["Division"].dropna().unique():
                    cur.execute("""
                        INSERT INTO divisions (divtype)
                        SELECT %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM divisions
                            WHERE divtype = %s
                        );
                    """, (clean(name), clean(name)))

                # 2. DEPARTMENTS
                departments = good[
                    ["Department", "DepartmentDescription"]
                ].drop_duplicates("Department")

                for _, r in departments.iterrows():
                    cur.execute("""
                        INSERT INTO departments
                            (departmentname, departmentdescription)
                        SELECT %s, %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM departments
                            WHERE departmentname = %s
                        );
                    """, (
                        clean(r["Department"]),
                        clean(r["DepartmentDescription"]),
                        clean(r["Department"])
                    ))

                # 3. JOB TITLES
                for title in good["JobTitle"].dropna().unique():
                    cur.execute("""
                        INSERT INTO jobtitles (jobtitle)
                        SELECT %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM jobtitles
                            WHERE jobtitle = %s
                        );
                    """, (clean(title), clean(title)))

                # 4. LOOKUP DICTIONARIES
                cur.execute("SELECT divtype, divisionid FROM divisions")
                division_ids = dict(cur.fetchall())

                cur.execute(
                    "SELECT departmentname, departmentid FROM departments"
                )
                department_ids = dict(cur.fetchall())

                cur.execute("SELECT jobtitle, jobtitleid FROM jobtitles")
                jobtitle_ids = dict(cur.fetchall())

                # 5. PERSONAL DETAILS
                people = good.drop_duplicates("SocialSecurity")

                for _, r in people.iterrows():
                    cur.execute("""
                        INSERT INTO personaldetails
                            (ssn, name, honorific, highestdegree,
                             email, phonenumber, birthdate,
                             perstreet, percity, perstate, perzip)
                        SELECT
                            %s, %s, %s, %s, %s, %s,
                            %s, %s, %s, %s, %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM personaldetails
                            WHERE ssn = %s
                        );
                    """, (
                        clean(r["SocialSecurity"]),
                        clean(r["Name"]),
                        None,
                        clean(r["HighestDegree"]),
                        clean(r["Email"]),
                        clean(r["PhoneNumber"]),
                        clean(r["Birthdate"]),
                        clean(r["PerStreetAddress"]),
                        clean(r["PerCity"]),
                        clean(r["PerState"]),
                        clean(r["PerZip"]),
                        clean(r["SocialSecurity"])
                    ))

                # 6. DIVISION LOCATIONS
                locations = good.drop_duplicates("Division")

                for _, r in locations.iterrows():
                    did = division_ids[r["Division"]]

                    cur.execute("""
                        INSERT INTO divisionlocations
                            (divisionid, divstreet, divcity,
                             divstate, divzip, divisionphone)
                        SELECT %s, %s, %s, %s, %s, %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM divisionlocations
                            WHERE divisionid = %s
                        );
                    """, (
                        clean(did),
                        clean(r["DivStreetAddress"]),
                        clean(r["DivCity"]),
                        clean(r["DivState"]),
                        clean(r["DivZip"]),
                        clean(r["DivisionPhone"]),
                        clean(did)
                    ))

                # 7. JOB DESCRIPTIONS
                descriptions = good[
                    ["Division", "JobTitle", "JobDescription"]
                ].drop_duplicates(["Division", "JobTitle"])

                for _, r in descriptions.iterrows():
                    did = division_ids[r["Division"]]
                    jid = jobtitle_ids[r["JobTitle"]]

                    cur.execute("""
                        INSERT INTO jobdescriptions
                            (jobtitleid, divisionid, description)
                        SELECT %s, %s, %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM jobdescriptions
                            WHERE jobtitleid = %s
                              AND divisionid = %s
                        );
                    """, (
                        clean(jid),
                        clean(did),
                        clean(r["JobDescription"]),
                        clean(jid),
                        clean(did)
                    ))

                # 8. EMPLOYEE DETAILS (LAST)
                for _, r in good.iterrows():
                    cur.execute("""
                        INSERT INTO employeedetails
                            (employeeid, ssn, departmentid,
                             jobtitleid, divisionid, salary)
                        SELECT %s, %s, %s, %s, %s, %s
                        WHERE NOT EXISTS (
                            SELECT 1 FROM employeedetails
                            WHERE employeeid = %s
                        );
                    """, (
                        clean(r["EmployeeID"]),
                        clean(r["SocialSecurity"]),
                        clean(department_ids[r["Department"]]),
                        clean(jobtitle_ids[r["JobTitle"]]),
                        clean(division_ids[r["Division"]]),
                        clean(r["Salary"]),
                        clean(r["EmployeeID"])
                    ))

        print("Employee load completed and committed.")

    finally:
        conn.close()



In [38]:
with engine.connect() as conn:
    limits = pd.read_sql(text("""
        SELECT
            table_name,
            column_name,
            character_maximum_length
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND character_maximum_length = 29;
    """), conn)

display(limits)

,table_name,column_name,character_maximum_length


In [39]:
long_titles = good[
    good["JobTitle"].str.len() > 29
][["JobTitle"]].drop_duplicates()

print("Job titles exceeding 29 characters:", len(long_titles))
display(long_titles)

Job titles exceeding 29 characters: 3


,JobTitle
7,Human Resources Manager Level 1
134,Human Resources Manager Level 2
501,Human Resources Manager Level 3


### A4. Run it twice and reconcile

Run your loader, record the 7 table counts, run it **again**, record them again. Show one table with, for every table: the count you expected from the CSV, the count after run 1, the count after run 2, and whether all three agree.

Then run the given NaN check below it. Every count must be 0.

In [41]:
# A4: Run twice and reconcile

TABLES = [
    "divisions",
    "departments",
    "jobtitles",
    "personaldetails",
    "divisionlocations",
    "jobdescriptions",
    "employeedetails"
]

def table_counts():
    with engine.connect() as conn:
        return {
            t: conn.execute(
                text(f"SELECT COUNT(*) FROM {t}")
            ).scalar()
            for t in TABLES
        }

# First load
load_employees(good)
after_run_1 = table_counts()

# Second load
load_employees(good)
after_run_2 = table_counts()

# Expected counts from CSV
expected = {
    "divisions": good["Division"].nunique(),
    "departments": good["Department"].nunique(),
    "jobtitles": good["JobTitle"].nunique(),
    "personaldetails": good["SocialSecurity"].nunique(),
    "divisionlocations": good["Division"].nunique(),
    "jobdescriptions": len(
        good[["Division", "JobTitle"]].drop_duplicates()
    ),
    "employeedetails": good["EmployeeID"].nunique()
}

recon = pd.DataFrame({
    "expected_from_csv": expected,
    "after_run_1": after_run_1,
    "after_run_2": after_run_2
})

recon["ok"] = (
    (recon["expected_from_csv"] == recon["after_run_1"])
    & (recon["after_run_1"] == recon["after_run_2"])
)

display(recon)


Employee load completed and committed.
Employee load completed and committed.


,expected_from_csv,after_run_1,after_run_2,ok
divisions,5,5,5,True
departments,6,6,6,True
jobtitles,51,51,51,True
personaldetails,14321,14321,14321,True
divisionlocations,5,5,5,True
jobdescriptions,235,235,235,True
employeedetails,14321,14321,14321,True


In [42]:
# Given: NaN check. Every count must be 0. If not, a value reached the database without clean().
with engine.connect() as conn:
    nan_check = {
        "salary = NaN":    conn.execute(text("SELECT COUNT(*) FROM employeedetails WHERE salary = 'NaN';")).scalar(),
        "name = 'NaN'":    conn.execute(text("SELECT COUNT(*) FROM personaldetails WHERE name = 'NaN';")).scalar(),
        "jobtitle = 'NaN'": conn.execute(text("SELECT COUNT(*) FROM jobtitles WHERE jobtitle = 'NaN';")).scalar(),
    }
nan_check

{'salary = NaN': 0, "name = 'NaN'": 0, "jobtitle = 'NaN'": 0}

### A5. Load report (write in this cell, 4 to 6 sentences)

- How many rows you read, loaded and rejected, and the reason for each kind of reject.
- Which loading pattern you used for `divisionlocations`, `jobdescriptions` and `employeedetails`, and why.
- One real problem you hit while building the loader (the error message or wrong count) and how you found and fixed it.


_Your answer here._

# Part B: SQL

Write your own SQL for each question in the code cell under it, using the `q()` helper, and leave the result showing. Rules for every question:
- Filter by **names** (`'Engineering'`, `'Austin'`), never by an id number you looked up.
- Never type a number that the database can compute (an average, a count).
- Show names, not ids, wherever a person would read the result.

In [43]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

**Q1.** Headcount by department: each department's name and number of employees, largest first.

In [44]:
# Q1
q("""
SELECT
    d.departmentname AS department,
    COUNT(e.employeeid) AS headcount
FROM departments d
LEFT JOIN employeedetails e
    ON d.departmentid = e.departmentid
GROUP BY d.departmentid, d.departmentname
ORDER BY headcount DESC;
""")

,department,headcount
0,Operations,10581
1,Administrative,2000
2,Engineering,1085
3,Human Resources,645
4,Management,5
5,Executive,5


**Q2.** Engineering employees earning less than $100,000: name, job title and salary, highest salary first.

In [45]:
# Q2
q("""
SELECT
    p.name,
    j.jobtitle,
    e.salary
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
JOIN jobtitles j ON e.jobtitleid = j.jobtitleid
JOIN departments d ON e.departmentid = d.departmentid
WHERE d.departmentname = 'Engineering'
  AND e.salary < 100000
ORDER BY e.salary DESC;
""")

,name,jobtitle,salary
0,Jennifer Bonnie Duffy,Engineer Level 4,99980.47
1,Luis Christopher Holder,Engineer Level 3,99890.86
2,Michelle Mary Davis,Engineer Level 1,99860.47
3,Lisa Crystal Austin,Engineer Level 3,99838.28
4,Catherine Deanna Hunter,Engineer Level 4,99820.89
...,...,...,...
391,Brian Shawn Parrish,Engineer Level 1,81480.18
392,Billy Daniel Jenkins,Engineer Level 1,81478.14
393,Kaitlyn Amanda Allen,Engineer Level 1,81471.96
394,Jeremiah Jared Marshall,Engineer Level 1,81424.51


**Q3.** Human Resources employees who work at the division located in **Austin**: name, job title and division. Find the division through its city; do not type the division's name.

In [46]:
# Q3
q("""
SELECT
    p.name,
    j.jobtitle,
    v.divtype AS division
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
JOIN jobtitles j ON e.jobtitleid = j.jobtitleid
JOIN departments d ON e.departmentid = d.departmentid
JOIN divisions v ON e.divisionid = v.divisionid
JOIN divisionlocations l ON v.divisionid = l.divisionid
WHERE d.departmentname = 'Human Resources'
  AND l.divcity = 'Austin'
ORDER BY p.name;
""")

,name,jobtitle,division
0,Alan Jordan Rivera,Human Resources Specialist 3,South Division
1,Alexandra Jody Reese,Human Resources Specialist 1,South Division
2,Andrew Rodney Williams,Human Resources Specialist 1,South Division
3,Angela Brenda Cook,Human Resources Specialist 4,South Division
4,Angela Megan Hernandez,Human Resources Manager Level 1,South Division
...,...,...,...
124,Victor Andrew Curry,Human Resources Manager Level 2,South Division
125,Walter Martin Hale,Human Resources Specialist 1,South Division
126,Wendy Amy Stokes,Human Resources Manager Level 1,South Division
127,Wendy Linda Reynolds,Human Resources Specialist 2,South Division


**Q4.** Human Resources only: for each division, the number of HR employees and their average salary (2 decimals), highest average first.

In [47]:
# Q4
q("""
SELECT
    v.divtype AS division,
    COUNT(e.employeeid) AS hr_headcount,
    ROUND(AVG(e.salary), 2) AS average_salary
FROM employeedetails e
JOIN departments d ON e.departmentid = d.departmentid
JOIN divisions v ON e.divisionid = v.divisionid
WHERE d.departmentname = 'Human Resources'
GROUP BY v.divisionid, v.divtype
ORDER BY average_salary DESC;
""")

,division,hr_headcount,average_salary
0,East Division,129,53445.78
1,North Division,129,53045.74
2,Central Division,129,52765.92
3,West Division,129,52742.41
4,South Division,129,52523.17


**Q5.** Engineering employees whose salary is at or above the average Engineering salary: name, job title, salary. The average must be computed inside the query.

In [48]:
# Q5
q("""
SELECT
    p.name,
    j.jobtitle,
    e.salary
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
JOIN jobtitles j ON e.jobtitleid = j.jobtitleid
JOIN departments d ON e.departmentid = d.departmentid
WHERE d.departmentname = 'Engineering'
  AND e.salary >= (
      SELECT AVG(e2.salary)
      FROM employeedetails e2
      JOIN departments d2
          ON e2.departmentid = d2.departmentid
      WHERE d2.departmentname = 'Engineering'
  )
ORDER BY e.salary DESC;
""")

,name,jobtitle,salary
0,Lindsey Breanna Villa,Lead Engineering Manager,291487.46
1,Timothy Matthew Ortega,Lead Engineering Manager,262471.31
2,Jillian Danielle Bray,Lead Engineering Manager,231262.00
3,Rodney Kyle Anderson,Lead Engineering Manager,223912.18
4,Samantha Nicole Snyder,Lead Engineering Manager,208455.58
...,...,...,...
456,Michael Jeffrey Petersen,Engineer Level 4,108514.87
457,Douglas Mark Anderson,Engineer Level 1,108401.88
458,Eric Michael Velez,Engineer Level 1,108376.49
459,Patricia Jennifer Miller,Engineer Level 1,108323.75


**Q6.** Employees aged 50 or older versus under 50 (age as of today): for each group, the number of employees and the average salary. Leave out anyone with no birthdate.

In [49]:
# Q6
q("""
SELECT
    CASE
        WHEN p.birthdate <= CURRENT_DATE - INTERVAL '50 years'
            THEN '50 or older'
        ELSE 'Under 50'
    END AS age_group,
    COUNT(*) AS employee_count,
    ROUND(AVG(e.salary), 2) AS average_salary
FROM employeedetails e
JOIN personaldetails p ON e.ssn = p.ssn
WHERE p.birthdate IS NOT NULL
GROUP BY
    CASE
        WHEN p.birthdate <= CURRENT_DATE - INTERVAL '50 years'
            THEN '50 or older'
        ELSE 'Under 50'
    END
ORDER BY age_group;
""")

,age_group,employee_count,average_salary
0,50 or older,3257,46925.51
1,Under 50,11064,43871.23


**Q7.** For each division: its city and state, its headcount and its average salary, largest headcount first.

In [50]:
# Q7
q("""
SELECT
    v.divtype AS division,
    l.divcity AS city,
    l.divstate AS state,
    COUNT(e.employeeid) AS headcount,
    ROUND(AVG(e.salary), 2) AS average_salary
FROM divisions v
JOIN divisionlocations l
    ON v.divisionid = l.divisionid
LEFT JOIN employeedetails e
    ON v.divisionid = e.divisionid
GROUP BY
    v.divisionid,
    v.divtype,
    l.divcity,
    l.divstate
ORDER BY headcount DESC;
""")

,division,city,state,headcount,average_salary
0,South Division,Austin,TX,2868,45532.09
1,Central Division,Columbus,OH,2864,44412.83
2,West Division,Sacramento,CA,2863,44316.07
3,North Division,Springfield,IL,2863,44336.94
4,East Division,Albany,NY,2863,44229.72


**Q8.** Departments whose average salary is higher than the company-wide average salary, with that department average.

In [51]:
# Q8
q("""
SELECT
    d.departmentname AS department,
    ROUND(AVG(e.salary), 2) AS average_salary
FROM employeedetails e
JOIN departments d
    ON e.departmentid = d.departmentid
GROUP BY d.departmentid, d.departmentname
HAVING AVG(e.salary) > (
    SELECT AVG(salary)
    FROM employeedetails
)
ORDER BY average_salary DESC;
""")

,department,average_salary
0,Executive,756238.76
1,Management,227950.60
2,Engineering,108286.63
3,Human Resources,52904.60


**Q9.** The highest-paid employee in each department: department, name, job title and salary.

In [52]:
# Q9
q("""
WITH ranked_employees AS (
    SELECT
        d.departmentname AS department,
        p.name,
        j.jobtitle,
        e.salary,
        RANK() OVER (
            PARTITION BY d.departmentid
            ORDER BY e.salary DESC
        ) AS salary_rank
    FROM employeedetails e
    JOIN departments d
        ON e.departmentid = d.departmentid
    JOIN personaldetails p
        ON e.ssn = p.ssn
    JOIN jobtitles j
        ON e.jobtitleid = j.jobtitleid
)
SELECT
    department,
    name,
    jobtitle,
    salary
FROM ranked_employees
WHERE salary_rank = 1
ORDER BY department, name;
""")

,department,name,jobtitle,salary
0,Administrative,Christopher Justin Stanley,Secretary Level 8,55967.39
1,Engineering,Lindsey Breanna Villa,Lead Engineering Manager,291487.46
2,Executive,Mark Alexander Hill,Chief Executive Officer,1111605.37
3,Human Resources,Michelle Angela Conley,Lead HR Manager,192030.36
4,Management,Darryl Taylor Heath,Division Manager,249261.08
5,Operations,Veronica Rebecca Mclaughlin,Lead Operations Manager,293491.71


**Q10.** Data quality: every email address that belongs to more than one person, with how many people share it. Then answer in the markdown cell below.

In [53]:
# Q10
q("""
SELECT
    email,
    COUNT(DISTINCT ssn) AS number_of_people
FROM personaldetails
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(DISTINCT ssn) > 1
ORDER BY number_of_people DESC, email;
""")

,email,number_of_people
0,anthony_thomas_200@KLB.com,2
1,cheryl_peterson_139@KLB.com,2
2,christopher_young_182@KLB.com,2
3,james_smith_138@KLB.com,2
4,jessica_anderson_140@KLB.com,2
5,joseph_davis_144@KLB.com,2
6,kevin_jones_171@KLB.com,2
7,luis_davis_161@KLB.com,2
8,melissa_moore_147@KLB.com,2
9,michael_cook_148@KLB.com,2


**Q10, part 2 (2 to 3 sentences):** Look at the rows Q10 returned. Is a shared email a data error, or could it be legitimate? KLB wants to use email as each employee's login. What would you check or change in the data or the schema before that is safe?


A shared email address is not necessarily a data error because multiple employees might legitimately use a shared departmental or organizational email address. However, if the company plans to use email addresses as unique login identifiers, shared emails could cause authentication and security problems. I would verify whether the shared addresses belong to separate employees, assign unique login identifiers where necessary, and enforce a UNIQUE constraint on the chosen login field after resolving any duplicates.

# Part C: Dashboard

Your dashboard is `app.py`, a separate file. Build it from the specification in the instructions PDF, run it with `streamlit run app.py`, and put your screenshots and answers in the PDF. Nothing for Part C goes in this notebook.

# Part D: Industry scenarios

These are the kind of incidents data engineers handle at large companies. There is no single right answer. Answer each one in the same cell, under the question, in 3 to 5 sentences. Refer to **your own** loader, SQL or `app.py` wherever you can: a specific answer (the line, the query, the setting) beats a general one.

**D1. The job said SUCCESS.** Your loader runs every night. For months it has rejected a handful of rows each night. One Monday it rejects about 4,100 of 14,000 rows, the job still reports SUCCESS, and the dashboard's headcount drops by almost a third. Nobody notices for two days. The cause: the HR system started sending salaries as `"$85,000.00"` instead of `85000`.

1. Look at how **your** loader converts `Salary`. What would it do with `"$85,000.00"`, and where would those rows end up? If your loader would behave differently from the story above, say how.
2. Why did nothing fail?
3. What would you add to the loader so that next time this stops the run or alerts someone?


In my loader, `pd.to_numeric(df["Salary"], errors="coerce")` would convert salaries formatted as `"$85,000.00"` into `NaN` because the dollar sign and comma are not recognized as numeric values. My validation code would mark those records as invalid and put them into `rejects`, while the remaining valid records could still be loaded successfully. Nothing failed because the invalid rows were handled by the validation logic rather than raising an exception, and the loader did not have a threshold for the number of rejected records. To prevent this, I would add a rejection-rate check that stops the load if too many records are rejected, send an alert, and log the reason for each rejection. I would also standardize salary formats by removing dollar signs and commas before converting the values to numbers.

**D2. Two numbers for the same thing.** Finance's report and your dashboard both read the same `EmployeeDB`, but they show different average salaries for Engineering. Your manager asks which one is wrong.

1. Give three legitimate reasons the two numbers could differ without either query having a bug. At least one reason must come from how your own `app.py` works.
2. How would you settle which number the company should use?


The average Engineering salary could differ because my Streamlit dashboard applies division, department, and salary-range filters, while Finance might calculate the average across all Engineering employees. Another reason could be that Finance uses a different definition of compensation, such as including bonuses, while my dashboard averages the `salary` column. A third possibility is that my dashboard caches SQL query results for 600 seconds, so it may temporarily display older data than Finance's report. To resolve the disagreement, I would compare both reports' filters, salary definitions, data refresh times, and employee counts, then work with Finance to establish a consistent definition of average salary. Once agreed upon, I would document that definition and use the same calculation across both reports.

**D3. From 14,000 rows to 50 million.** After KLB acquires another company, the nightly file grows from about 14,000 rows to 50 million.

1. Time one run of your loader on your own data (for example, wrap one of the runs in A4 with `time.perf_counter()`) and report the time.
2. What in your loader would break or slow down first at 50 million rows? Point to the specific part of your code and explain why it gets slow.
3. What would you change?


I tested my loader using 14,321 employee records, and one additional run took **[INSERT YOUR MEASURED TIME] seconds**. If the dataset increased to 50 million records, the biggest performance issue would be the `for _, r in good.iterrows()` loop in my `load_employees()` function, which processes employees individually and executes a separate SQL statement for each record. This would cause millions of database operations and make the ETL process extremely slow, while loading the entire CSV into pandas could also consume a large amount of memory. To improve performance, I would process the data in chunks, use PostgreSQL's `COPY` command or batch inserts instead of inserting one row at a time, and add appropriate indexes for employee IDs and foreign keys. I would also use incremental loading so the pipeline only processes new or updated employees instead of reloading all 50 million records every night.

In [54]:
import time

start = time.perf_counter()

load_employees(good)

elapsed = time.perf_counter() - start

print(f"Loader runtime: {elapsed:.2f} seconds")

Employee load completed and committed.
Loader runtime: 17.60 seconds
